# Documentation technique de l'acquisition

**Autrice :** Faty Barbosa Monteiro

**Date :** 23 octobre 2026

**Client :** CLIENT001

**Lu après :** `documentation/03_data_acquisition.ipynb`

Cette note décrit les programmes qui exécutent le registre des sources. La note d'acquisition fixe les jeux et les fournisseurs. Le classeur `tests/acquisition/phase_03_validation.ipynb` contrôle les fichiers obtenus. Il ne décrit pas les programmes.


## 1. Objet

La commande `python -m src.pipeline.refresh` publie les jeux dus, puis charge en base ceux dont le chargement est autorisé. La suite de la même commande prépare les séries en francs et poursuit les calculs. Cette note s'arrête au chargement.

Quatre sources sont actives.

| Source | Fournisseur | Jeux |
|---|---|---|
| `SRC-001` | iShares / BlackRock | `DS-000`, `DS-001`, `DS-002` |
| `SRC-002` | Yahoo Finance | `DS-003` |
| `SRC-003` | Banque nationale suisse | `DS-004`, `DS-005` |
| `SRC-004` | Rapports annuels, rapports de durabilité et sites des sociétés | `DS-006` |

Aucun de ces programmes n'attribue un statut d'éligibilité aux Objectifs de développement durable (ODD).


## 2. Configuration

`src/config.py` fixe les chemins du projet et la devise de reporting, le franc. Les scripts importent ces chemins. Ils ne les réécrivent pas.

Le registre `audit/source_registry.xlsx` est l'autorité de chaque jeu : identifiant, source, méthode de production, connecteur, adresse, paramètres, jeux amont, nom de fichier, répertoire, fréquence, chargement, contrôle qualité, chronologie, activité. Les moteurs lisent ce registre. Ils n'y inscrivent pas d'identifiant en dur.



## 3. Commande et calendrier

`src/pipeline/refresh.py` ordonne les jeux actifs de façon que chaque jeu amont soit publié avant le jeu qui le lit. L'option `--force` republie tous les jeux actifs, sans tenir compte de la fréquence.

La fréquence est lue dans le registre. L'horloge est UTC. Un succès déjà enregistré pour le même fichier, dans le même compartiment de calendrier, rend le jeu non dû. Le fichier présent est alors conservé.

| Fréquence | Compartiment | Jeux |
|---|---|---|
| Daily | le jour | `DS-003` |
| Monthly | le mois | `DS-004`, `DS-005` |
| Periodic | le mois | `DS-000`, `DS-001` |
| Annual | l'année | `DS-006` |
| On demand | pas de compartiment | `DS-002` |

Un jeu à la demande est dû lorsqu'aucune publication réussie de son fichier n'existe, ou lorsqu'un de ses jeux amont est republé dans la même exécution.

Une acquisition en échec n'annule pas les autres. Si le fichier d'un jeu amont manque, le jeu qui en dépend est sauté.


## 4. Moteurs

La méthode de production choisit le moteur.

| Méthode | Programme | Rôle |
|---|---|---|
| Download | `src/acquisition/extract_download.py` | Télécharge le fichier HTTP, le vérifie, le publie de façon atomique |
| API | `src/acquisition/extract_api.py` | Appelle le connecteur nommé dans le registre |
| Package | `src/acquisition/extract_package.py` | Même enchaînement, pour un paquet Python |
| Preparation | le constructeur du fichier de sortie | Transforme un ou plusieurs jeux déjà publiés |

Les connecteurs sont dans `src/acquisition/connectors/`. `base.py` fixe le contrat. `resolver.py` charge le module dont le nom est la colonne Connector. Le connecteur valide ses paramètres, interroge le fournisseur et rend une table fidèle à la réponse. Il ne connaît ni l'identifiant de source, ni le répertoire de sortie, ni le journal, ni la base. Le moteur s'occupe de l'ordonnancement, de la publication et du journal.

Deux connecteurs sont enregistrés : `snb` et `yfinance`.

Les préparations appelées par la commande sont :

| Fichier | Programme |
|---|---|
| `security_candidates.csv` | `src/preparation/securities_candidates.py` |
| `securities_master.csv` | `src/preparation/securities_master.py` |
| `odd_assessment.csv` | `src/preparation/odd_assessment.py` |

`src/preparation/exchange_mapping.py` n'est pas un jeu. C'est la table qui traduit le libellé de place du fichier iShares vers le suffixe Yahoo et la devise de cotation. Quand le code iShares n'est pas le code de la place, une ligne tenue à la main donne ce code, avec la page qui le publie. Une place absente de la table reste sans symbole. Elle n'est pas devinée.


## 5. iShares — DS-000 et DS-001

`DS-000`, Benchmark Holdings, est un Download périodique. Il n'a pas de connecteur. Le fichier reçu est `data/01_raw/benchmark/benchmark_holdings.xls`. Il est chargé en base. Les tables publiées sont celles des feuilles `Holdings`, `Performance`, `Historical` et `Distributions`. Le bandeau du fichier et la feuille `Disclaimers` ne sont pas des tables.

Le document est du SpreadsheetML, malgré l'extension `.xls`. La préparation lit la feuille nommée dans les paramètres, `Holdings`. Elle conserve la classe d'actif `Equity` et déduplique sur `Ticker` et `Exchange`.

`DS-001`, Security Candidates, est cette préparation. Le fichier est `data/02_processed/securities/security_candidates.csv`. Il est chargé en base. Le contrôle qualité est désactivé. Les attributs retenus sont `Ticker`, `Name`, `Location`, `Exchange`, `Currency` et `Asset Class`.

L'indice MSCI ACWI Sustainable Development Index (Net) n'a pas de cours propre. Il n'est pas téléchargé. `DS-000` est la feuille de positions du fonds iShares MSCI Global Sustainable Development Goals ETF.


## 6. Référentiel — DS-002

`DS-002`, Security Master, est la préparation de `DS-001`. La source reste `SRC-001`. Le fichier est `data/02_processed/securities/securities_master.csv`. Il est chargé en base et soumis au contrôle qualité.

La préparation lit le code et la place de chaque candidat. La table des places en tire `yahoo_symbol` et `listing_currency`. Si une ligne d'écart existe pour ce code et cette place, le symbole part du code de la place. Une place absente de la table ne reçoit pas de symbole inventé. Le symbole Yahoo et la devise de cotation sont renseignés lorsque la table des places les fournit.

Les phases suivantes lisent ce symbole et cette devise.


## 7. Yahoo Finance — DS-003

`DS-003`, Market Prices, est un Package quotidien. Le connecteur est `yfinance`. Il n'a pas d'adresse d'accès. Le jeu amont est `DS-002`. Le symbole lu est `yahoo_symbol`. Les paramètres demandent l'historique maximal en séance quotidienne, avec `auto_adjust` à faux, les opérations du titre demandées, sans réparation des cours et en conservant les séances vides. Le fichier est `data/01_raw/market/market_prices.csv`. Il est chargé en base. Le contrôle qualité et la chronologie sont activés.

Le fichier conserve le cours d'ouverture, le plus haut, le plus bas, le cours de clôture, le cours ajusté, le volume, ainsi que les colonnes `dividends` et `stock_splits` renvoyées par le fournisseur. `auto_adjust` à faux laisse le cours de clôture non ajusté. Le rendement du mandat est ce cours de clôture, en fin de mois, converti en francs. Il n'ajoute pas la colonne des dividendes et n'utilise pas le cours ajusté. La conversion en francs et la correction d'échelle de certaines cotations sont faites après le chargement, dans la préparation des séries. Elles ne sont pas faites dans le connecteur.

### Identité du symbole

`yahoo_symbol` reste le symbole de référence. Le connecteur peut voir un autre symbole chez Yahoo. Il ne le substitue pas au symbole configuré. Le téléchargement automatique est réservé au symbole configuré lorsque son identité est confirmée.

La recherche exacte passe avant toute alternative.

| Résultat | Téléchargement | Suite |
|---|---|---|
| `DIRECT_VALIDATED` | autorisé | le symbole configuré est retenu |
| `SEARCH_RESOLVED` | refusé | un candidat alternatif unique est conservé pour revue |
| `IDENTITY_AMBIGUOUS` | refusé | les réponses ne suffisent pas |
| `IDENTITY_NOT_FOUND` | refusé | aucune identité exploitable après une réponse techniquement valide |

Quatre preuves sont lues séparément : le symbole exact, le type de titre, la place, le nom. Chacune vaut `MATCH`, `CONTRADICTION` ou `UNKNOWN`. `UNKNOWN` constate une preuve insuffisante. Une contradiction matérielle empêche `DIRECT_VALIDATED`. Aucun score global de similarité des noms n'est calculé.

L'absence d'un symbole sur le chemin testé, signalée par une réponse de type introuvable, permet encore la recherche alternative. Une erreur d'authentification, une limite de débit, un délai, une erreur réseau ou une réponse inexploitable arrête le traitement de ce titre. Cette erreur n'est pas convertie en `IDENTITY_NOT_FOUND`.

Chaque ligne acquise conserve `configured_symbol`, `resolved_symbol` et `symbol_resolution_status`. Le connecteur ne réécrit pas le référentiel. Il ne promeut pas un `SEARCH_RESOLVED` au rang de symbole de référence.


## 8. Banque nationale suisse — DS-004 et DS-005

Les deux jeux passent par le connecteur `snb` et l'adresse `https://data.snb.ch/api/cube`. Le connecteur lit d'abord la date d'édition du cube, puis le cube filtré. La valeur reste dans l'unité publiée.

`DS-004`, FX Rates, est une API mensuelle. Le cube est `devkum`. La fréquence demandée est `M1`, depuis janvier 1970. La langue est l'anglais et le format est JSON. Les devises demandées sont les `listing_currency` de `DS-002`, traduites par `snb_currency_codes` vers le code du cube. Le franc, devise de reporting, n'est pas demandé au cube. Une devise de cotation absente de cette table n'est pas envoyée. Le cube ne publie pas `ILS`, `INR`, `SAR` ni `TWD`. Un titre qui porte l'une de ces devises n'a pas de cours de change. Le fichier est `data/01_raw/fx/fx_rates.csv`. Il est chargé en base. Le contrôle qualité et la chronologie sont activés. Un code tel que `USD1` désigne un dollar et `JPY100` désigne cent yens. La division par cent n'est pas faite à l'acquisition.

`DS-005`, SARON, est une API mensuelle sans jeu amont. Le cube est `zimoma`. La série demandée est `SARON`, depuis janvier 2009. Le fichier est `data/01_raw/macro/saron.csv`. Il est chargé en base. Le contrôle qualité et la chronologie sont activés. La valeur est le taux publié, en pour cent. Le connecteur ne le transforme pas en taux mensuel.


## 9. Registre ODD — DS-006

`DS-006`, ODD Assessment, est une préparation annuelle de `DS-002`. Il n'a ni connecteur, ni adresse. `SRC-004` n'a pas d'endpoint unique : les rapports et les sites sont lus société par société. Le fichier est `data/01_raw/odd/odd_assessment.csv`. Il est chargé en base. Le contrôle qualité est désactivé.

La préparation ouvre une ligne par titre du référentiel, identifié par `Ticker` et `Exchange`. Une nouvelle exécution conserve les lignes déjà présentes et ajoute les titres qui manquent. Le programme n'écrit ni l'ODD, ni la contribution, ni la controverse, ni le statut. Le statut est saisi à partir du document nommé. Les valeurs prévues sont `Eligible`, `Not_Eligible`, `Indeterminate` et `Excluded`. Tant que le statut n'est pas écrit, la ligne ne reçoit pas de poids. Il n'y a pas de colonne `Data Quality`. La phase 4 vérifie le fichier : une ligne par titre du référentiel, aucun doublon sur `Ticker` et `Exchange`, un statut parmi les quatre mots officiels, et, pour une ligne Eligible, l'ODD, les deux contributions, la preuve, la source et la controverse. Elle compte les numéros d'ODD imprimés, les controverses encore ouvertes et les parts de chiffre d'affaires énoncées. Ce constat est écrit dans la note du contrôle et dans le journal technique. Il n'entre pas dans le score, et il ne réécrit pas le fichier. `Positive` est une phrase sur l'activité. Ce n'est pas une preuve.

FIGI et ISIN ne sont pas des colonnes du registre. Aucun document ouvert ne les a imprimés, et l'identité est `Ticker` + `Exchange`.

Le contrat de colonnes est celui de `src/preparation/odd_assessment.py`. Une colonne absente de ce contrat empêche une nouvelle préparation d'accepter le fichier.


## 10. Journaux

Deux journaux sont tenus.

Le journal technique, `logs/technical.log`, est écrit par `src/utils/logger.py`. Il reçoit le démarrage, la validation du registre, l'avertissement et l'erreur. Le fichier tourne à 5 Mo et conserve cinq archives. Une ligne a la forme : horodatage, niveau, message.

Le journal d'acquisition est la table SQLite `acquisition_log`, écrite par `src/acquisition/acquisition_logger.py`. C'est le registre des exécutions. Le classeur `audit/acquisition_log.xlsx` en est l'export, produit par `src/reporting/export_acquisition_log.py`. L'export rapproche chaque fichier du registre par le nom du fichier. Un fichier préparé qui n'est pas un fichier de sortie du registre reste sans rapprochement. Cet écart est un avertissement de l'export. Ce n'est pas un échec d'acquisition.

Chaque exécution reçoit un identifiant `RUN-AAAAMMJJ-HHMMSS-XXXXXXXX`. La table enregistre le type de processus (`Acquisition` ou `Preparation`), la date UTC, le fichier, l'emplacement, le statut, le nombre de lignes et une note. Les statuts admis sont `Planned`, `Success`, `Partial Success` et `Failed`. L'identifiant de source et l'identifiant de jeu sont repris à l'export depuis le registre. Ils ne sont pas des colonnes de la table.


## 11. Base

`src/database/load_database.py` charge chaque jeu actif dont Load Enabled est vrai. La table est écrite dans une table d'étape, comptée, puis publiée. Le catalogue `dataset_registry` retient l'emplacement, le dernier statut d'acquisition, la date de ce statut, la date de chargement et la présence de la table.

Les tables de données sont `benchmark_holdings`, `benchmark_performance`, `benchmark_historical`, `benchmark_distributions`, `security_candidates`, `securities_master`, `market_prices`, `fx_rates`, `saron` et `odd_assessment`.

La base sert aux contrôles et aux requêtes. Les calculs lisent les fichiers préparés après le chargement : le cours en francs, le SARON traité et la liste des titres dont la devise de cotation a une série de la Banque nationale suisse. Après le chargement, la commande complète aussi la devise de cotation d'un symbole du référentiel qui ne l'aurait pas encore. Ce fichier de devises n'est pas un jeu du registre.


## 12. Limite de la collecte

La collecte automatisée s'arrête aux sept jeux du registre.

Les scores ESG ne sont pas demandés. Les exclusions du tabac, des armes conventionnelles, du charbon thermique et des jeux d'argent attendent une source nommée, société par société. Aucun connecteur ne les invente. L'indice stratégique n'est pas téléchargé. Le pétrole, le gaz, le nucléaire et l'alcool ne font pas l'objet d'une liste automatique.

Les connecteurs présents sont `snb` et `yfinance`.


# Dictionnaire des termes de cette note

Ce dictionnaire reprend les termes employés dans cette note. Il n'en ajoute pas.

**Connecteur.** Programme qui parle au fournisseur nommé dans le registre. Il rend la réponse. Il ne choisit pas le répertoire ni le statut d'éligibilité.

**Cube.** Tableau publié par la Banque nationale suisse. `devkum` fournit les cours de change. `zimoma` fournit le SARON.

**DIRECT_VALIDATED.** Le symbole Yahoo configuré est confirmé. Seul ce résultat autorise le téléchargement automatique.

**Fréquence.** Rythme inscrit au registre. Elle décide si le jeu est dû.

**IDENTITY_AMBIGUOUS.** Yahoo a répondu, sans identité assez sûre pour télécharger.

**IDENTITY_NOT_FOUND.** Aucune identité exploitable après une réponse techniquement valide.

**Journal d'acquisition.** Table `acquisition_log`. Le classeur Excel en est l'export.

**Journal technique.** Fichier `logs/technical.log`. Il suit l'exécution des programmes.

**Méthode de production.** Download, API, Package ou Preparation. Elle choisit le moteur.

**ODD.** Objectifs de développement durable. Le registre les attend des documents de la société. Le programme n'écrit pas le statut.

**Préparation.** Jeu construit à partir d'un jeu déjà publié, sans nouvel appel fournisseur, sauf la table interne des places.

**RAW.** Fichier conservé tel que le fournisseur l'a publié.

**Référentiel.** `DS-002`, une ligne par titre, avec le symbole Yahoo et la devise de cotation.

**SARON.** Taux au jour le jour en francs, publié par la Banque nationale suisse dans le cube `zimoma`.

**SEARCH_RESOLVED.** Un symbole Yahoo alternatif unique a été vu. Il est gardé pour revue et n'est pas téléchargé à la place du symbole configuré.

**Source Registry.** Classeur qui nomme les sources et les jeux. Les moteurs l'exécutent.
